# Analysing Patch Counts

Looking at number of correct and overfitting patches in the classical (Petke) dataset.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [ ]:
TOOL_COLUMNS = [
	"Avatar",
	"FixMiner",
	"Nopol",
	"SimFix",
	"TBar",
	"dynamoth",
	"GenProg",
	"Kali",
	"MatRepair",
	"KPAR",
]

PATCH_TYPES = ("correct", "overfitting")

TOOL_NAME_MAP = {
	"avatar": "Avatar",
	"fixminer": "FixMiner",
	"nopol": "Nopol",
	"simfix": "SimFix",
	"tbar": "TBar",
	"dynamoth": "dynamoth",
	"jgenprog": "GenProg",
	"genprog": "GenProg",
	"jkali": "Kali",
	"kali": "Kali",
	"jmutrepair": "MatRepair",
	"matrepair": "MatRepair",
	"kpar": "KPAR",
}


def parse_patch_filename(file_path: Path) -> tuple[str, str] | None:
	stem_parts = file_path.stem.split("_")
	if len(stem_parts) < 4:
		return None

	tool_name = TOOL_NAME_MAP.get(stem_parts[2].lower())
	if tool_name is None:
		return None

	return f"{stem_parts[0]}-{stem_parts[1]}", tool_name


def create_patch_counts_dataframe(dataset_dir: Path) -> pd.DataFrame:
	"""Return Petke patch counts indexed by bug name and patch type.

	``dataset_dir`` must contain the ``correct`` and ``overfitting`` directories.
	The result has one row per observed bug and patch type, tool-count columns, and
	a ``Total`` column that sums the tool columns.
	"""
	records: list[dict[str, str]] = []
	for patch_type in PATCH_TYPES:
		patch_dir = dataset_dir / patch_type
		if not patch_dir.is_dir():
			raise NotADirectoryError(
				f"Patch type directory does not exist: {patch_dir.resolve()}"
			)

		for patch_file in sorted(patch_dir.glob("*.diff")):
			parsed = parse_patch_filename(patch_file)
			if parsed is None:
				continue

			bug_name, tool_name = parsed
			records.append(
				{
					"Bug Name": bug_name,
					"Patch Type": patch_type,
					"Tool": tool_name,
				}
			)

	if not records:
		index = pd.MultiIndex.from_arrays([[], []], names=["Bug Name", "Patch Type"])
		return pd.DataFrame(columns=[*TOOL_COLUMNS, "Total"], index=index, dtype=int)

	counts = pd.DataFrame.from_records(records)
	table = counts.groupby(["Bug Name", "Patch Type", "Tool"]).size().unstack(
		"Tool", fill_value=0
	)
	table = table.reindex(columns=TOOL_COLUMNS, fill_value=0).astype(int)
	table["Total"] = table.sum(axis=1)
	return table.sort_index()

In [2]:
patch_counts = create_patch_counts_dataframe(Path("datasets/petke/patches_by_time/patches_8h_deduplicated"))
patch_counts

Tool                     Avatar  FixMiner  Nopol  SimFix  TBar  dynamoth  \
Bug Name    Patch Type                                                     
Chart-1     correct           1         0      0       3     1         0   
            overfitting       6         0      0       0     8         3   
Chart-11    correct           1         1      0       0     1         0   
Chart-12    overfitting       0         0      0       1     0         0   
Chart-13    overfitting       0         1      1       0     2         0   
Chart-15    overfitting       0         1      0       0     1         0   
Chart-17    overfitting       0         0      1       0     0         0   
Chart-18    overfitting       0         0      0       1     0         0   
Chart-20    correct           0         0      0       1     1         0   
Chart-24    correct           1         1      0       0     1         0   
Chart-25    overfitting       5         0      1       0     1         2   
Chart-26    overfitting       0         0      1       0     2         0   
Chart-4     correct           0         2      0       0     3         0   
            overfitting       0         0      0       0     1         0   
Chart-5     overfitting       0         0      1       0     0         1   
Chart-7     correct           0         0      0       1     0         0   
            overfitting       8         0      0       2    12         0   
Chart-8     correct           0         0      0       0     1         0   
Chart-9     correct           0         0      0       0     1         0   
            overfitting       0         0     10       0     0         0   
Closure-10  overfitting       0         0      0       0     0         0   
Closure-115 overfitting       0         0      0       1     0         0   
Closure-126 correct           0         0      0       0     0         2   
            overfitting       2         0      4       0     3         1   
Closure-129 overfitting       0         0      0       0     0         1   
Closure-13  overfitting       0         0      0       0     0         0   
Closure-14  correct           0         0      0       1     0         0   
Closure-19  overfitting       0         0      0       0     0         1   
Closure-2   correct           0         0      0       0     1         0   
            overfitting       0         0      0       0     2         0   
Closure-21  overfitting       1         0      1       0     2         1   
Closure-22  overfitting       0         0      0       0     1         0   
Closure-46  correct           1         0      0       0     1         0   
            overfitting       0         0      0       0     0         2   
Closure-48  overfitting       0         0      0       0     0         1   
Closure-57  correct           0         0      0       2     0         0   
            overfitting       0         0      0       2     0         0   
Closure-62  correct           1         0      0       2     1         0   
            overfitting       5         0      1       2     4         1   
Closure-73  correct           1         0      0       1     0         0   
Closure-86  correct           0         0      0       0     1         0   
Lang-10     correct           2         0      0       0     1         0   
            overfitting       6         0      0       0     1         0   
Lang-33     correct           0         0      0       2     1         0   
            overfitting       0         0      0       0     2         0   
Lang-43     correct           0         0      0       2     0         0   
            overfitting       0         0      0       0     1         0   
Lang-44     overfitting       1         0      0       0     2         0   
Lang-45     overfitting       0         0      0       0     1         0   
Lang-46     overfitting       0         0      1       0     0         0   
Lang-51     overfitting       1         0      2

In [15]:
# APR-tool columns; exclude the aggregate Total column
tool_columns = [column for column in patch_counts.columns if column != "Total"]

# True when a tool generated at least one patch for a bug/patch-type row
has_patch = patch_counts[tool_columns].gt(0)

# Count distinct tools that generated at least one patch per bug.
# Grouping with `any()` takes the union across correct and overfitting rows.
tools_per_bug = (
    has_patch
    .groupby(level="Bug Name")
    .any()
    .sum(axis=1)
    .astype(int)
)

# Identify which patch types exist for each bug
patch_types_per_bug = (
    patch_counts
    .reset_index()[["Bug Name", "Patch Type"]]
    .drop_duplicates()
    .groupby("Bug Name")["Patch Type"]
    .agg(set)
)
print(patch_types_per_bug)

# Define the requested bug subsets
bug_subsets = {
    "All bugs": patch_types_per_bug.index,
    "Only correct patches": patch_types_per_bug[
        patch_types_per_bug == {"correct"}
    ].index,
    "Only overfitting patches": patch_types_per_bug[
        patch_types_per_bug == {"overfitting"}
    ].index,
    "Correct and overfitting patches": patch_types_per_bug[
        patch_types_per_bug == {"correct", "overfitting"}
    ].index,
}

# Calculate the summary statistics
summary = pd.DataFrame(
    {
        subset_name: tools_per_bug.loc[bug_names].agg(
            ["count", "median", "min", "max"]
        )
        for subset_name, bug_names in bug_subsets.items()
    }
).T

summary = summary.rename(
    columns={
        "count": "Number of bugs",
        "median": "Median number of APR tools",
        "min": "Minimum number of APR tools",
        "max": "Maximum number of APR tools",
    }
)

summary["Number of bugs"] = summary["Number of bugs"].astype(int)
summary["Median number of APR tools"] = summary[
    "Median number of APR tools"
].astype(int)
summary["Minimum number of APR tools"] = summary[
    "Minimum number of APR tools"
].astype(int)
summary["Maximum number of APR tools"] = summary[
    "Maximum number of APR tools"
].astype(int)

summary

Bug Name
Chart-1        {correct, overfitting}
Chart-11                    {correct}
Chart-12                {overfitting}
Chart-13                {overfitting}
Chart-15                {overfitting}
Chart-17                {overfitting}
Chart-18                {overfitting}
Chart-20                    {correct}
Chart-24                    {correct}
Chart-25                {overfitting}
Chart-26                {overfitting}
Chart-4        {correct, overfitting}
Chart-5                 {overfitting}
Chart-7        {correct, overfitting}
Chart-8                     {correct}
Chart-9        {correct, overfitting}
Closure-10              {overfitting}
Closure-115             {overfitting}
Closure-126    {correct, overfitting}
Closure-129             {overfitting}
Closure-13              {overfitting}
Closure-14                  {correct}
Closure-19              {overfitting}
Closure-2      {correct, overfitting}
Closure-21              {overfitting}
Closure-22              {overfitting}
Clo

,Number of bugs,Median number of APR tools,Minimum number of APR tools,Maximum number of APR tools
All bugs,92,2,1,8
Only correct patches,16,2,1,3
Only overfitting patches,47,2,1,7
Correct and overfitting patches,29,4,1,8


In [4]:
patch_totals_by_bug = (
    patch_counts["Total"]
    .unstack("Patch Type", fill_value=0)
    .reindex(columns=["overfitting", "correct"], fill_value=0)
    .rename(
        columns={
            "overfitting": "overfitting_patches",
            "correct": "correct_patches",
        }
    )
    .rename_axis("bug_name")
    .reset_index()
)

patch_totals_by_bug.head(10)

Patch Type,bug_name,overfitting_patches,correct_patches
0,Chart-1,21,8
1,Chart-11,0,3
2,Chart-12,1,0
3,Chart-13,5,0
4,Chart-15,4,0
5,Chart-17,1,0
6,Chart-18,1,0
7,Chart-20,0,2
8,Chart-24,0,3
9,Chart-25,13,0


In [5]:
patch_totals_by_bug['total_patches'] = patch_totals_by_bug['overfitting_patches'] + patch_totals_by_bug['correct_patches']

In [6]:
def stats(df):
    # median of "total_patches" column
    print("Number of bugs:", len(df))
    print("Median total patches:", df['total_patches'].median())
    print("Max total patches:", df['total_patches'].max())
    print("Min total patches:", df['total_patches'].min())

In [7]:
bugs_with_no_overfitting = patch_totals_by_bug[patch_totals_by_bug['overfitting_patches'] == 0]
print("Bugs with no overfitting patches:")
stats(bugs_with_no_overfitting)

Bugs with no overfitting patches:
Number of bugs: 16
Median total patches: 2.0
Max total patches: 4
Min total patches: 1


In [8]:
bugs_with_no_overfitting

Patch Type,bug_name,overfitting_patches,correct_patches,total_patches
1,Chart-11,0,3,3
7,Chart-20,0,2,2
8,Chart-24,0,3,3
14,Chart-8,0,1,1
21,Closure-14,0,1,1
30,Closure-73,0,2,2
31,Closure-86,0,1,1
43,Lang-6,0,1,1
47,Math-11,0,3,3
55,Math-34,0,2,2


In [9]:
bugs_with_overfitting = patch_totals_by_bug[(patch_totals_by_bug['overfitting_patches'] > 0) & (patch_totals_by_bug['correct_patches'] > 0)]
print("Bugs with overfitting patches:")
stats(bugs_with_overfitting)

Bugs with overfitting patches:
Number of bugs: 29
Median total patches: 11.0
Max total patches: 100
Min total patches: 3


In [10]:
bugs_with_overfitting.head(30)

Patch Type,bug_name,overfitting_patches,correct_patches,total_patches
0,Chart-1,21,8,29
11,Chart-4,1,5,6
13,Chart-7,37,1,38
15,Chart-9,10,1,11
18,Closure-126,16,5,21
23,Closure-2,2,1,3
26,Closure-46,10,4,14
28,Closure-57,2,2,4
29,Closure-62,15,5,20
32,Lang-10,7,3,10
